# <font color="Green">**Notebook Purpose and Required Data**</font>

This notebook constructs the final patient-level dataframe used for the simple linear regression analyses. It integrates treatment-group assignments, demographic attributes, baseline clinical measurements, and pre-2019 comorbidity indicators into a single regression-ready table (`slr_df`). The notebook performs the following tasks:

1. **Loads group-level patient files:**  
   Reads in the CSV files corresponding to each treatment trajectory group (e.g., Monotherapy, Dual Therapy, Complex Therapy, GLP-1 Therapy, Variant Therapy, Early Dropout Group).  
   The `patient_ids` column is converted from its serialized string form back into Python lists.

2. **Merges demographic information:**  
   Incorporates age (calculated as of 2019), sex, race/ethnicity, and patient regional location from the demographics dataset.

3. **Extracts baseline 2019 clinical measurements:**  
   Identifies each patient’s earliest HbA1c and BMI measurements within calendar year 2019.

4. **Adds pre-2019 comorbidity indicators:**  
   Loads the modified comorbidity table and merges heart failure and chronic kidney disease indicators (`had_HF_2019`, `had_CKD_2019`).

5. **Outputs a unified regression dataset:**  
   Produces a single consolidated dataframe (`slr_df`) suitable for downstream modeling.

---

### <font color="Red">Data Required</font>

The following files and data objects must already be present in the working directory.  
Examples of how to create each of these inputs, excluding the treatment-group pkl files, can be found in other notebooks within the **Data Preprocessing** folder. The treatment-group files are created after analyzing clustering results and assigning clinically relevant groups, examples of their creation can be found in the 'ClusteringAnalysis.ipynb' file in the 'Analysis' folder.

- **Treatment-group PKL files**  
  Each must contain:  
  - `cluster_id`  
  - `patient_ids`
  These dictionaries define patient membership for each treatment trajectory group.

- **Demographics file (`patient_demographics.csv`)**  
  Must include:  
  - `patient_id`  
  - `year_of_birth`  
  - `sex`  
  - `race_ethnicity` (or `race/ethnicity`, before renaming)  
  - `patient_regional_location`

- **HbA1c laboratory results (`lab_results.csv`)**  
  Must include:  
  - `patient_id`  
  - `date`  
  - `lab_result_num_val`

- **BMI vital signs (`BMI_vital_signs.csv`)**  
  Must include:  
  - `patient_id`  
  - `date`  
  - `value`

- **Patient Comorbidity data (`patient_comorbidities.csv`)**  
  Must include:  
  - `patient_id`  
  - `HF`  
  - `CKD`
  - `Date`  

All files must reference the same patient cohort and use consistent `patient_id` identifiers to allow successful merging across datasets.


In [ ]:
import pandas as pd
import numpy as np
import pickle
import ast

##<font color="black">**Read In Data**</font>

### Reading in standard patient data

In [ ]:
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
medication_info = pd.read_csv('/content/medication_info.csv')
lab_results = pd.read_csv('/content/lab_results.csv')
BMI_vital_signs = pd.read_csv('/content/BMI_vital_signs.csv')

### Reading in the treatment group data


In [ ]:
with open('/content/monotherapy_patients.pkl', 'rb') as f:
    monotherapy_patients = pickle.load(f)

with open('/content/dual_therapy_patients.pkl', 'rb') as f:
    dual_therapy_patients = pickle.load(f)

with open('/content/complex_therapy_patients.pkl', 'rb') as f:
    complex_therapy_patients = pickle.load(f)

with open('/content/GLP_1_therapy_patients.pkl', 'rb') as f:
    GLP_1_therapy_patients = pickle.load(f)

with open('/content/variant_therapy_patients.pkl', 'rb') as f:
    variant_therapy_patients = pickle.load(f)

# We decided to include the early dropout group in this table despite not including this
# group in our final cohort. When the any regression work was done, this group is emitted.
with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pickle.load(f)

### Reading in comorbidity data

In [ ]:
patient_comorbidities = pd.read_csv('/content/patient_comorbidities.csv')
patient_comorbidities.head()

##<font color="black">**Creating a modified comorbidities table**</font>

In this section, we construct a table for use in the regression model. The resulting dataset, patient_comorbidities_pre2019, contains three columns—patient_id, had_HF, and had_CKD—and identifies whether each patient had a diagnosis of heart failure or chronic kidney disease prior to, or at the start of, the study period. This table provides a clear baseline indicator of comorbidity status.

In [ ]:
# --- Config
INDEX_DATE = pd.Timestamp('2019-01-01')

# --- Start from raw tables
pc = patient_comorbidities.copy()
demo = patient_demographics[['patient_id']].drop_duplicates()

# --- Ensure types
pc['date'] = pd.to_datetime(pc['date'], errors='coerce')
pc['HF']  = pc['HF'].fillna(False).astype(bool)
pc['CKD'] = pc['CKD'].fillna(False).astype(bool)

# --- Restrict to baseline (<= 2019-01-01)
pc_pre = pc.loc[pc['date'].notna() & (pc['date'] <= INDEX_DATE), ['patient_id', 'HF', 'CKD']]

# --- Collapse to one row per patient with flags (any pre-2019 HF/CKD)
pre_flags = (
    pc_pre
    .groupby('patient_id', as_index=False)
    .agg(had_HF=('HF', 'any'), had_CKD=('CKD', 'any'))
)

# --- Ensure every cohort patient appears exactly once; fill missing with False
patient_comorbidities_pre2019 = (
    demo
    .merge(pre_flags, on='patient_id', how='left')
    .fillna({'had_HF': False, 'had_CKD': False})
)

# --- Enforce boolean dtype (fillna can sometimes upcast)
patient_comorbidities_pre2019['had_HF']  = patient_comorbidities_pre2019['had_HF'].astype(bool)
patient_comorbidities_pre2019['had_CKD'] = patient_comorbidities_pre2019['had_CKD'].astype(bool)

patient_comorbidities_pre2019.head()

##<font color="black">**Simple Linear Regression Table Creation**</font>

In [ ]:
# -----------------------------
# Config / constants
# -----------------------------
YEAR = 2019
START_2019 = pd.Timestamp('2019-01-01')
END_2019   = pd.Timestamp('2020-01-01')

# -----------------------------
# Helper: convert dict → DataFrame
# -----------------------------
def dict_to_group_df(group_dict):
    """
    Convert a treatment-group dictionary into a DataFrame with columns:
    ['cluster_id', 'patient_ids']

    group_dict: {cluster_id: list_of_patient_ids}
    """
    if not group_dict:
        return pd.DataFrame(columns=['cluster_id', 'patient_ids'])
    return pd.DataFrame({
        'cluster_id': list(group_dict.keys()),
        'patient_ids': list(group_dict.values())
    })

# -----------------------------
# 1) Build a unified (Group, group_rank, cluster_id, patient_id) table
# -----------------------------
def prep_group_df(df, group_name):
    """
    df: DataFrame with columns ['cluster_id', 'patient_ids'] where patient_ids is a list
    group_name: string like 'Monotherapy', 'Dual Therapy', etc. (no 'patients'/'df' in the name)
    """
    if df.empty:
        return pd.DataFrame(columns=['Group','group_rank','cluster_id','patient_id'])
    out = df.copy().reset_index(drop=True)
    out['Group'] = group_name
    out['group_rank'] = np.arange(1, len(out) + 1)
    out = out.explode('patient_ids').rename(columns={'patient_ids':'patient_id'})
    return out[['Group','group_rank','cluster_id','patient_id']]

groups_prep = [
    prep_group_df(dict_to_group_df(monotherapy_patients),      'Monotherapy'),
    prep_group_df(dict_to_group_df(dual_therapy_patients),     'Dual Therapy'),
    prep_group_df(dict_to_group_df(complex_therapy_patients),  'Complex Therapy'),
    prep_group_df(dict_to_group_df(GLP_1_therapy_patients),    'GLP-1 Therapy'),
    prep_group_df(dict_to_group_df(variant_therapy_patients),  'Variant Therapy'),
    prep_group_df(dict_to_group_df(early_dropout_patients),    'Early Dropout Group'),
]

group_long = pd.concat(groups_prep, ignore_index=True)

# -----------------------------
# 2) Demographics (age, sex, race_ethnicity, patient_regional_location)
# -----------------------------
demo = patient_demographics.copy()
# Compute age as of 2019 (no normalization per your instruction)
demo['age'] = YEAR - demo['year_of_birth']
# Standardize race/ethnicity column name
demo = demo.rename(columns={'race/ethnicity': 'race_ethnicity'})
demo = demo[['patient_id', 'age', 'sex', 'race_ethnicity', 'patient_regional_location']]

# -----------------------------
# 3) HbA1C_2019: earliest value within 2019
# Assumption: lab_results already represents HbA1c tests (no test-name filter provided).
# -----------------------------
labs = lab_results.copy()
labs['date'] = pd.to_datetime(labs['date'], errors='coerce')
labs_2019 = labs.loc[(labs['date'] >= START_2019) & (labs['date'] < END_2019)]
hbA1c_2019 = (
    labs_2019
    .sort_values('date')
    .groupby('patient_id', as_index=False)
    .first()[['patient_id', 'lab_result_num_val']]
    .rename(columns={'lab_result_num_val': 'HbA1C_2019'})
)

# -----------------------------
# 4) BMI_2019: earliest value within 2019
# -----------------------------
bmi = BMI_vital_signs.copy()
bmi['date'] = pd.to_datetime(bmi['date'], errors='coerce')
bmi_2019 = bmi.loc[(bmi['date'] >= START_2019) & (bmi['date'] < END_2019)]
BMI_2019 = (
    bmi_2019
    .sort_values('date')
    .groupby('patient_id', as_index=False)
    .first()[['patient_id', 'value']]
    .rename(columns={'value': 'BMI_2019'})
)

# -----------------------------
# 5) Pre-2019 comorbidities → flags for 2019 regression
# -----------------------------
comor = patient_comorbidities_pre2019.copy()
comor = comor.rename(columns={'had_HF':'had_HF_2019', 'had_CKD':'had_CKD_2019'})
# Ensure boolean dtype (in case of NA)
comor['had_HF_2019']  = comor['had_HF_2019'].fillna(False).astype(bool)
comor['had_CKD_2019'] = comor['had_CKD_2019'].fillna(False).astype(bool)

# -----------------------------
# 6) Merge everything into slr_df
# -----------------------------
slr_df = (
    group_long
    .merge(demo, on='patient_id', how='left')
    .merge(hbA1c_2019, on='patient_id', how='left')
    .merge(BMI_2019, on='patient_id', how='left')
    .merge(comor[['patient_id','had_HF_2019','had_CKD_2019']], on='patient_id', how='left')
)

# If some patients are missing comorbidity rows (shouldn’t happen given your pre-2019 table covers all),
# still safe to fill NA → False.
slr_df['had_HF_2019']  = slr_df['had_HF_2019'].fillna(False).astype(bool)
slr_df['had_CKD_2019'] = slr_df['had_CKD_2019'].fillna(False).astype(bool)

# Final column order
slr_df = slr_df[[
    'Group', 'group_rank', 'cluster_id', 'patient_id',
    'age', 'sex', 'race_ethnicity', 'patient_regional_location',
    'HbA1C_2019', 'BMI_2019', 'had_HF_2019', 'had_CKD_2019'
]]

# No duplicate (patient, cluster) rows
assert slr_df[['patient_id','cluster_id']].drop_duplicates().shape[0] == slr_df.shape[0]

slr_df.head()

Export to CSV

In [ ]:
slr_df.to_csv('slr_df.csv')